# Vivacity Modelling Ready Inputs

Converted from the original Python workflow script so the dissertation repository uses notebook-based workflow artefacts.


In [ ]:
#!/usr/bin/env python3
"""Build Vivacity modelling-ready datasets from the integrity gate."""

from __future__ import annotations

import json
from html import escape
from pathlib import Path

import pandas as pd


BASE = Path("/Users/lu_nanxi/CASA/Dissertation_Data")
CUTOFF_DIR = BASE / "Vivacity_full_day_cutoff_20260526"
DAILY_INPUT = CUTOFF_DIR / "control/vivacity_treated_and_control_daily_merged.csv"
GATE_INPUT = CUTOFF_DIR / "integrity_gate/vivacity_qa_inclusion_table.csv"
OUT_DIR = CUTOFF_DIR / "modelling_ready"
TABLE_DIR = OUT_DIR / "tables"
PLOT_DIR = OUT_DIR / "plots"
QA_DIR = OUT_DIR / "quality_checks"
AVAILABILITY_THRESHOLD = 80


COUNT_COLS = [
    "Car",
    "Pedestrian",
    "Cyclist",
    "Motorbike",
    "Bus",
    "OGV1",
    "OGV2",
    "LGV",
    "active_travel_total",
    "motorised_total",
]


def apply_confirmed_intervention_dates(df: pd.DataFrame) -> pd.DataFrame:
    if "intervention_date" not in df.columns:
        raise ValueError("Prepared input data must include an intervention_date column.")

    out = df.copy()
    out["intervention_date"] = pd.to_datetime(out["intervention_date"], errors="coerce")
    scheme_key = out["analysis_scheme_id"].astype(str).str.strip()
    missing_schemes = sorted(scheme_key[out["intervention_date"].isna()].dropna().unique().tolist())
    if missing_schemes:
        raise ValueError(
            "Prepared input data has missing intervention_date values for: "
            f"{missing_schemes}"
        )
    return out


def parse_bool(series: pd.Series, default: bool = False) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(default)
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame]:
    daily = pd.read_csv(DAILY_INPUT, low_memory=False)
    gate = pd.read_csv(GATE_INPUT, low_memory=False)

    daily["date"] = pd.to_datetime(daily["date"], format="mixed", errors="coerce")
    daily = apply_confirmed_intervention_dates(daily)
    gate["first_reliable_date"] = pd.to_datetime(gate["first_reliable_date"], errors="coerce")
    gate["intervention_date_gate"] = pd.to_datetime(gate["intervention_date"], errors="coerce")

    for col in COUNT_COLS + ["data_availability_percent_min", "data_availability_percent_mean"]:
        if col in daily.columns:
            daily[col] = pd.to_numeric(daily[col], errors="coerce")

    gate_cols = [
        "dataset_role",
        "analysis_scheme_id",
        "countline_id",
        "first_reliable_date",
        "usable_pre_days",
        "usable_post_days",
        "treated_use_in_causal_models",
        "control_use_pragmatic_causal",
        "control_use_strict_causal",
        "control_use_descriptive_only",
        "inclusion_tier",
        "strict_decision_reason",
        "pragmatic_decision_reason",
    ]
    gate_small = gate[gate_cols].copy()
    for col in [
        "treated_use_in_causal_models",
        "control_use_pragmatic_causal",
        "control_use_strict_causal",
        "control_use_descriptive_only",
    ]:
        gate_small[col] = parse_bool(gate_small[col])

    merged = daily.merge(
        gate_small,
        on=["dataset_role", "analysis_scheme_id", "countline_id"],
        how="left",
        validate="many_to_one",
    )
    return merged, gate


def add_quality_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["availability_missing_bool"] = (
        parse_bool(out["availability_missing"], default=True)
        | out["data_availability_percent_min"].isna()
    )
    out["low_availability_bool"] = (
        parse_bool(out["low_availability"], default=False)
        | (out["data_availability_percent_min"] < AVAILABILITY_THRESHOLD).fillna(False)
    )
    out["data_error_bool"] = parse_bool(out["data_error_any"], default=False)
    out["after_first_reliable_date"] = (
        out["first_reliable_date"].notna() & (out["date"] >= out["first_reliable_date"])
    )
    out["quality_ok_day"] = (
        out["date"].notna()
        & out["intervention_date"].notna()
        & ~out["availability_missing_bool"]
        & ~out["low_availability_bool"]
        & ~out["data_error_bool"]
    )
    out["use_row_after_gate"] = out["after_first_reliable_date"] & out["quality_ok_day"]
    out["main_causal_countline"] = (
        (out["dataset_role"].eq("treated") & out["treated_use_in_causal_models"].fillna(False))
        | (out["dataset_role"].eq("control") & out["control_use_pragmatic_causal"].fillna(False))
    )
    out["main_causal_row"] = out["use_row_after_gate"] & out["main_causal_countline"]
    out["descriptive_row"] = out["use_row_after_gate"]
    out["week_start"] = out["date"] - pd.to_timedelta(out["date"].dt.weekday, unit="D")
    out["month_start"] = out["date"].values.astype("datetime64[M]")
    out["analysis_period"] = out.apply(
        lambda row: "pre" if row["date"] < row["intervention_date"] else "post", axis=1
    )
    out["role_label"] = out["dataset_role"].map({"treated": "Treated", "control": "Control"})
    out.loc[
        out["dataset_role"].eq("control") & out["control_use_pragmatic_causal"].fillna(False),
        "role_label",
    ] = "Valid control"
    out.loc[
        out["dataset_role"].eq("control") & out["control_use_descriptive_only"].fillna(False),
        "role_label",
    ] = "Descriptive-only control"
    return out


def aggregate_weekly(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    countline_group = [
        "dataset_role",
        "role_label",
        "analysis_scheme_id",
        "scheme_id",
        "scheme_name",
        "road_group",
        "treated_road_group",
        "installation_month",
        "intervention_date",
        "inclusion_tier",
        "countline_id",
        "countline_name",
        "route_type",
        "candidate_lsoa21cd",
        "candidate_lsoa21nm",
        "candidate_lad",
        "first_reliable_date",
        "control_use_pragmatic_causal",
        "control_use_strict_causal",
        "treated_use_in_causal_models",
        "week_start",
    ]
    weekly_countline = (
        df.groupby(countline_group, dropna=False)
        .agg(
            observed_days=("date", "nunique"),
            pedestrian=("Pedestrian", "sum"),
            cyclist=("Cyclist", "sum"),
            active_travel_total=("active_travel_total", "sum"),
            motorised_total=("motorised_total", "sum"),
            data_availability_percent_min=("data_availability_percent_min", "min"),
            data_availability_percent_mean=("data_availability_percent_mean", "mean"),
        )
        .reset_index()
    )
    weekly_countline["active_per_observed_day"] = (
        weekly_countline["active_travel_total"] / weekly_countline["observed_days"]
    )
    weekly_countline["pedestrian_per_observed_day"] = (
        weekly_countline["pedestrian"] / weekly_countline["observed_days"]
    )
    weekly_countline["cyclist_per_observed_day"] = weekly_countline["cyclist"] / weekly_countline[
        "observed_days"
    ]
    weekly_countline["analysis_period"] = weekly_countline.apply(
        lambda row: "pre" if row["week_start"] < row["intervention_date"] else "post", axis=1
    )

    scheme_group = [
        "analysis_scheme_id",
        "scheme_name",
        "treated_road_group",
        "installation_month",
        "intervention_date",
        "dataset_role",
        "role_label",
        "inclusion_tier",
        "week_start",
    ]
    weekly_scheme = (
        weekly_countline.groupby(scheme_group, dropna=False)
        .agg(
            countlines=("countline_id", "nunique"),
            observed_countline_days=("observed_days", "sum"),
            pedestrian=("pedestrian", "sum"),
            cyclist=("cyclist", "sum"),
            active_travel_total=("active_travel_total", "sum"),
            motorised_total=("motorised_total", "sum"),
            min_availability=("data_availability_percent_min", "min"),
            mean_availability=("data_availability_percent_mean", "mean"),
        )
        .reset_index()
    )
    weekly_scheme["active_per_countline_day"] = (
        weekly_scheme["active_travel_total"] / weekly_scheme["observed_countline_days"]
    )
    weekly_scheme["pedestrian_per_countline_day"] = (
        weekly_scheme["pedestrian"] / weekly_scheme["observed_countline_days"]
    )
    weekly_scheme["cyclist_per_countline_day"] = (
        weekly_scheme["cyclist"] / weekly_scheme["observed_countline_days"]
    )
    weekly_scheme["analysis_period"] = weekly_scheme.apply(
        lambda row: "pre" if row["week_start"] < row["intervention_date"] else "post", axis=1
    )
    return weekly_countline, weekly_scheme


def write_outputs(df: pd.DataFrame, gate: pd.DataFrame) -> dict:
    for path in [OUT_DIR, TABLE_DIR, PLOT_DIR, QA_DIR]:
        path.mkdir(parents=True, exist_ok=True)

    missing_gate = df[df["first_reliable_date"].isna()][
        ["dataset_role", "analysis_scheme_id", "countline_id", "countline_name"]
    ].drop_duplicates()
    missing_gate.to_csv(QA_DIR / "modelling_missing_or_no_first_reliable_rows.csv", index=False)

    main_daily = df[df["main_causal_row"]].copy()
    descriptive_daily = df[df["descriptive_row"]].copy()

    main_weekly_countline, main_weekly_scheme = aggregate_weekly(main_daily)
    desc_weekly_countline, desc_weekly_scheme = aggregate_weekly(descriptive_daily)

    main_daily.to_csv(TABLE_DIR / "vivacity_main_causal_daily.csv", index=False)
    descriptive_daily.to_csv(TABLE_DIR / "vivacity_descriptive_daily.csv", index=False)
    main_weekly_countline.to_csv(TABLE_DIR / "vivacity_main_causal_weekly_countline.csv", index=False)
    main_weekly_scheme.to_csv(TABLE_DIR / "vivacity_main_causal_weekly_scheme_role.csv", index=False)
    desc_weekly_countline.to_csv(TABLE_DIR / "vivacity_descriptive_weekly_countline.csv", index=False)
    desc_weekly_scheme.to_csv(TABLE_DIR / "vivacity_descriptive_weekly_scheme_role.csv", index=False)

    countline_summary = (
        df.groupby(
            [
                "dataset_role",
                "analysis_scheme_id",
                "countline_id",
                "countline_name",
                "inclusion_tier",
                "first_reliable_date",
                "treated_use_in_causal_models",
                "control_use_pragmatic_causal",
                "control_use_strict_causal",
                "control_use_descriptive_only",
            ],
            dropna=False,
        )
        .agg(
            raw_rows=("date", "size"),
            rows_after_quality_gate=("use_row_after_gate", "sum"),
            main_causal_rows=("main_causal_row", "sum"),
            descriptive_rows=("descriptive_row", "sum"),
            first_model_date=("date", lambda s: s[df.loc[s.index, "use_row_after_gate"]].min()),
            last_model_date=("date", lambda s: s[df.loc[s.index, "use_row_after_gate"]].max()),
        )
        .reset_index()
    )
    countline_summary.to_csv(QA_DIR / "modelling_countline_inclusion_summary.csv", index=False)

    summary = {
        "daily_input": str(DAILY_INPUT),
        "gate_input": str(GATE_INPUT),
        "output_dir": str(OUT_DIR),
        "source_rows": int(len(df)),
        "main_causal_daily_rows": int(len(main_daily)),
        "descriptive_daily_rows": int(len(descriptive_daily)),
        "main_causal_countlines": int(
            main_daily[["dataset_role", "analysis_scheme_id", "countline_id"]].drop_duplicates().shape[0]
        ),
        "descriptive_countlines": int(
            descriptive_daily[
                ["dataset_role", "analysis_scheme_id", "countline_id"]
            ].drop_duplicates().shape[0]
        ),
        "main_causal_schemes": sorted(main_daily["analysis_scheme_id"].astype(str).unique().tolist()),
        "outputs": {
            "main_causal_daily": str(TABLE_DIR / "vivacity_main_causal_daily.csv"),
            "descriptive_daily": str(TABLE_DIR / "vivacity_descriptive_daily.csv"),
            "main_causal_weekly_scheme_role": str(
                TABLE_DIR / "vivacity_main_causal_weekly_scheme_role.csv"
            ),
            "descriptive_weekly_scheme_role": str(
                TABLE_DIR / "vivacity_descriptive_weekly_scheme_role.csv"
            ),
            "countline_summary": str(QA_DIR / "modelling_countline_inclusion_summary.csv"),
        },
    }
    (OUT_DIR / "vivacity_modelling_ready_summary.json").write_text(
        json.dumps(summary, indent=2), encoding="utf-8"
    )
    return summary


def _svg_line_chart(
    data: pd.DataFrame,
    metric: str,
    ylabel: str,
    filename: str,
    schemes: list[str],
    colors: dict[str, str],
) -> None:
    width = 1180
    panel_height = 300
    margin_left = 92
    margin_right = 34
    margin_top = 54
    margin_bottom = 44
    legend_x = width - 270
    total_height = panel_height * len(schemes)
    svg: list[str] = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{total_height}" viewBox="0 0 {width} {total_height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<style>text{font-family:Arial, Helvetica, sans-serif; fill:#222;} .axis{stroke:#444;stroke-width:1;} .grid{stroke:#ddd;stroke-width:1;} .series{fill:none;stroke-width:2.2;} .tick{font-size:11px;fill:#555;} .title{font-size:16px;font-weight:700;} .label{font-size:12px;fill:#444;}</style>',
    ]

    for idx, scheme in enumerate(schemes):
        panel_y = idx * panel_height
        sub = data[data["analysis_scheme_id"].astype(str).eq(scheme)].copy()
        if sub.empty:
            continue
        x_min = sub["week_start"].min()
        x_max = sub["week_start"].max()
        if x_min == x_max:
            x_max = x_min + pd.Timedelta(days=7)
        y_max = float(sub[metric].max())
        y_max = max(y_max, 1.0)
        y_max *= 1.08
        plot_x0 = margin_left
        plot_x1 = width - margin_right
        plot_y0 = panel_y + margin_top
        plot_y1 = panel_y + panel_height - margin_bottom
        plot_w = plot_x1 - plot_x0
        plot_h = plot_y1 - plot_y0

        def sx(value: pd.Timestamp) -> float:
            return plot_x0 + ((value - x_min).days / max((x_max - x_min).days, 1)) * plot_w

        def sy(value: float) -> float:
            return plot_y1 - (float(value) / y_max) * plot_h

        svg.append(f'<text class="title" x="{plot_x0}" y="{panel_y + 25}">Scheme {escape(scheme)}: treated vs valid controls</text>')
        svg.append(f'<text class="label" x="{plot_x0}" y="{panel_y + 43}">{escape(ylabel)}</text>')

        for tick_i in range(5):
            y_val = y_max * tick_i / 4
            y = sy(y_val)
            svg.append(f'<line class="grid" x1="{plot_x0}" x2="{plot_x1}" y1="{y:.1f}" y2="{y:.1f}"/>')
            svg.append(f'<text class="tick" x="{plot_x0 - 8}" y="{y + 4:.1f}" text-anchor="end">{y_val:.0f}</text>')

        tick_count = 6
        for tick_i in range(tick_count):
            frac = tick_i / (tick_count - 1)
            dt = x_min + pd.Timedelta(days=round((x_max - x_min).days * frac))
            x = sx(dt)
            svg.append(f'<line class="axis" x1="{x:.1f}" x2="{x:.1f}" y1="{plot_y1}" y2="{plot_y1 + 5}"/>')
            svg.append(f'<text class="tick" x="{x:.1f}" y="{plot_y1 + 20}" text-anchor="middle">{dt.strftime("%Y-%m")}</text>')

        svg.append(f'<line class="axis" x1="{plot_x0}" x2="{plot_x1}" y1="{plot_y1}" y2="{plot_y1}"/>')
        svg.append(f'<line class="axis" x1="{plot_x0}" x2="{plot_x0}" y1="{plot_y0}" y2="{plot_y1}"/>')

        intervention = sub["intervention_date"].dropna().min()
        if pd.notna(intervention) and x_min <= intervention <= x_max:
            x_int = sx(intervention)
            svg.append(f'<line x1="{x_int:.1f}" x2="{x_int:.1f}" y1="{plot_y0}" y2="{plot_y1}" stroke="#222" stroke-width="1.2" stroke-dasharray="5 5"/>')
            svg.append(f'<text class="tick" x="{x_int + 5:.1f}" y="{plot_y0 + 13}">intervention</text>')

        legend_y = panel_y + 25
        for leg_i, (role, role_df) in enumerate(sub.groupby("role_label")):
            role_df = role_df.sort_values("week_start")
            points = " ".join(
                f'{sx(row.week_start):.1f},{sy(getattr(row, metric)):.1f}'
                for row in role_df.itertuples(index=False)
                if pd.notna(getattr(row, metric))
            )
            color = colors.get(role, "#555")
            svg.append(f'<polyline class="series" points="{points}" stroke="{color}"/>')
            ly = legend_y + leg_i * 18
            svg.append(f'<line x1="{legend_x}" x2="{legend_x + 24}" y1="{ly}" y2="{ly}" stroke="{color}" stroke-width="2.5"/>')
            svg.append(f'<text class="label" x="{legend_x + 32}" y="{ly + 4}">{escape(role)}</text>')

    svg.append("</svg>")
    (PLOT_DIR / filename).write_text("\n".join(svg), encoding="utf-8")


def plot_weekly_main(weekly: pd.DataFrame) -> None:
    if weekly.empty:
        return
    plot_data = weekly[
        weekly["role_label"].isin(["Treated", "Valid control"])
    ].copy()
    schemes = sorted(plot_data["analysis_scheme_id"].astype(str).unique())
    metrics = [
        ("active_per_countline_day", "Active travel counts per countline-day", "main_weekly_active_travel.svg"),
        ("pedestrian_per_countline_day", "Pedestrian counts per countline-day", "main_weekly_pedestrian.svg"),
        ("cyclist_per_countline_day", "Cyclist counts per countline-day", "main_weekly_cyclist.svg"),
    ]
    colors = {"Treated": "#1f77b4", "Valid control": "#d62728"}
    for metric, ylabel, filename in metrics:
        _svg_line_chart(plot_data, metric, ylabel, filename, schemes, colors)


def plot_observed_counts(weekly: pd.DataFrame) -> None:
    if weekly.empty:
        return
    plot_data = weekly[weekly["role_label"].isin(["Treated", "Valid control"])].copy()
    schemes = sorted(plot_data["analysis_scheme_id"].astype(str).unique())
    _svg_line_chart(
        plot_data,
        "countlines",
        "Observed countlines in weekly panel",
        "main_weekly_observed_countlines.svg",
        schemes,
        {"Treated": "#1f77b4", "Valid control": "#d62728"},
    )


def write_readme(summary: dict) -> None:
    text = f"""# Vivacity Modelling-Ready Inputs

Generated from the full-day cutoff dataset and the integrity-gate decisions.

## Main causal dataset

The main causal dataset keeps:

- treated countlines marked `treated_valid_for_causal_side`;
- control countlines marked `control_pragmatic_exploratory_causal` or `control_strict_seasonal_causal`;
- only rows on/after each countline's `first_reliable_date`;
- only rows with availability present, `dataAvailabilityPercent_min >= 80`, and no data error.

Strict seasonal controls are retained within the pragmatic control set, but the modelling notes should report that only two controls pass the stricter 365-day pre/post threshold.

## Descriptive dataset

The descriptive dataset keeps all countlines after their first reliable date and row-level quality filtering, including controls and treated schemes that are not suitable for causal comparison.

## Outputs

- `tables/vivacity_main_causal_daily.csv`
- `tables/vivacity_descriptive_daily.csv`
- `tables/vivacity_main_causal_weekly_countline.csv`
- `tables/vivacity_main_causal_weekly_scheme_role.csv`
- `tables/vivacity_descriptive_weekly_countline.csv`
- `tables/vivacity_descriptive_weekly_scheme_role.csv`
- `quality_checks/modelling_countline_inclusion_summary.csv`

## Summary

- Source rows: {summary['source_rows']}
- Main causal daily rows: {summary['main_causal_daily_rows']}
- Descriptive daily rows: {summary['descriptive_daily_rows']}
- Main causal countlines: {summary['main_causal_countlines']}
- Descriptive countlines: {summary['descriptive_countlines']}
- Main causal schemes: {', '.join(summary['main_causal_schemes'])}
"""
    (OUT_DIR / "README.md").write_text(text, encoding="utf-8")


def main() -> int:
    merged, gate = load_inputs()
    flagged = add_quality_flags(merged)
    summary = write_outputs(flagged, gate)
    weekly_main = pd.read_csv(TABLE_DIR / "vivacity_main_causal_weekly_scheme_role.csv", parse_dates=["week_start", "intervention_date"])
    plot_weekly_main(weekly_main)
    plot_observed_counts(weekly_main)
    write_readme(summary)
    print(json.dumps(summary, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
